# Complete retrospective Stage-2: strongest model first

Unexecuted operator notebook for R-only development plus separately sealed held-out P:0006 final evaluation. It invokes reviewed production CLIs for Variant A, streamed factored-bank construction/audit, the 200-step full-objective pilot, strongest-first training, feasibility-gated genuine-R evaluation when available, otherwise strict import of Gate01Private_8012a3f for P:0006 evaluation only, and later backward ablations. No P endpoint enters training or checkpoint selection and descriptor coupling remains disabled. One authorization launches only the strongest full model; a separate post-review flag controls all backward ablations.


In [ ]:
from pathlib import Path, PureWindowsPath
from collections import Counter
import copy, hashlib, importlib, json, os, re, subprocess, sys, tomllib

REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
BASE_COMMIT = '09605a0ce5a3c14d3e19ea7c719405d5cc5d35b3'
IMPLEMENTATION_REF = input('Reviewed unified implementation commit SHA: ').strip()
if re.fullmatch(r'[0-9a-f]{40}', IMPLEMENTATION_REF) is None:
    raise ValueError('Pin the exact externally reviewed 40-character implementation SHA.')
REPO_DIR = Path('/content/MRIxFields-stage2-unified')
if REPO_DIR.exists():
    raise FileExistsError('Start a fresh runtime; refusing to mutate an existing checkout.')
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', IMPLEMENTATION_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', IMPLEMENTATION_REF], cwd=REPO_DIR, check=True)
def git_text(*args):
    return subprocess.check_output(['git', *args], cwd=REPO_DIR, text=True).strip()
if git_text('rev-parse', 'HEAD') != IMPLEMENTATION_REF:
    raise RuntimeError('Detached implementation SHA mismatch.')
if subprocess.run(['git', 'merge-base', '--is-ancestor', BASE_COMMIT, 'HEAD'], cwd=REPO_DIR).returncode:
    raise RuntimeError('Unified implementation is not based on the reviewed merged main.')
if git_text('status', '--porcelain') or subprocess.run(['git', 'symbolic-ref', '-q', 'HEAD'], cwd=REPO_DIR, capture_output=True).returncode == 0:
    raise RuntimeError('Checkout must be detached and clean.')
print({'head': IMPLEMENTATION_REF, 'base': BASE_COMMIT, 'clean': True, 'detached': True})


In [ ]:
# Install only dependencies declared by this detached checkout and force its imports.
with (REPO_DIR / 'pyproject.toml').open('rb') as handle:
    project = tomllib.load(handle)['project']
optional = project['optional-dependencies']
requirements = list(dict.fromkeys([*project['dependencies'], *optional['evaluation'], *optional['official-evaluation']]))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *requirements], check=True)
SOURCE_DIR = str(REPO_DIR / 'src')
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONPATH'] = SOURCE_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, SOURCE_DIR)
for name in tuple(sys.modules):
    if name == 'fieldbridge' or name.startswith('fieldbridge.'):
        del sys.modules[name]
importlib.invalidate_caches()
import fieldbridge
if REPO_DIR not in Path(fieldbridge.__file__).resolve().parents:
    raise RuntimeError('FieldBridge import escaped the detached checkout.')
CLI_ENV = os.environ.copy()
CLI_ENV['PYTHONPATH'] = SOURCE_DIR + os.pathsep + CLI_ENV.get('PYTHONPATH', '')
if git_text('status', '--porcelain'):
    raise RuntimeError('Dependency installation changed the checkout.')
print({'fieldbridge_import': str(Path(fieldbridge.__file__).resolve()), 'requirements': requirements})


In [ ]:
# All operator paths are supplied once, before any scientific array is loaded.
from google.colab import drive
drive.mount('/content/drive')
FROZEN_SPLIT_V3_JSON = Path(input('Immutable split-v3 JSON: ').strip()).expanduser()
RETROSPECTIVE_DATA_ROOT = Path(input('Retrospective data root: ').strip()).expanduser()
FROZEN_STAGE1_VAE_CONFIG = Path(input('Frozen Stage-1 VAE config: ').strip()).expanduser()
FROZEN_VAE_CHECKPOINT = Path(input('Frozen VAE checkpoint: ').strip()).expanduser()
GATE01_RESULT_JSON = Path(input('Reviewed Gate 0.1 result JSON: ').strip()).expanduser()
R_PAIRED_EVALUATION_ARCHIVE_RAW = input('Optional reviewed R-paired producer-export archive root (used only if feasibility succeeds): ').strip()
GATE01_PRIVATE_ARCHIVE_ROOT = Path(input('Sealed Gate01Private_8012a3f archive root for P:0006 fallback: ').strip()).expanduser()
EXTERNAL_OUTPUT_ROOT = Path(input('New or exactly resumable external output root: ').strip()).expanduser()
EXPECTED_VAE_CONFIG_SHA256 = input('Frozen VAE config SHA-256: ').strip().lower()
EXPECTED_VAE_CHECKPOINT_SHA256 = input('Frozen VAE checkpoint SHA-256: ').strip().lower()
GPU_HOURLY_COST_USD = float(input('Effective GPU hourly cost in USD (use 0 for already-paid allocation): ').strip())
EXPECTED_SPLIT_V3_SHA256 = 'f6a19d7a31c4c3bb73edd92088ea078192e88ee4b276309bad81c548ab7f94d5'
EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256 = 'cbe885f73a307065418ea80296d6cfd6d634edeb3281f503cf90e149800409e7'
FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256 = '569a17b1316a47a0c95c42c649f4aab61f8fe8c9cf7d0582c411f544c9b23173'
EXPECTED_GATE01_RESULT_SHA256 = '454747cd3e4b1376855915244a7c40fe281b758150e86f584fbea96f94d531f5'
REVIEWED_WINDOWS_SOURCE_ROOT = r'D:\MRI_Field_2026\Data'
for label, value in {'VAE config': EXPECTED_VAE_CONFIG_SHA256, 'VAE checkpoint': EXPECTED_VAE_CHECKPOINT_SHA256}.items():
    if re.fullmatch(r'[0-9a-f]{64}', value) is None:
        raise ValueError(f'{label} requires an exact lowercase SHA-256.')
if not GATE01_PRIVATE_ARCHIVE_ROOT.is_dir(): raise FileNotFoundError('Gate01Private_8012a3f archive root is required for the fail-closed unpaired-corpus fallback.')
if not 0 <= GPU_HOURLY_COST_USD < 1000: raise ValueError('GPU hourly cost must be finite and non-negative.')


In [ ]:
# Deterministic split remap and byte-identical merged inventory arithmetic; no arrays open here.
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.photometry_factorization import all_photometry_domain_labels, assert_variant_a_external_path, sha256_file, sha256_json, sha256_text, write_json_atomic
from fieldbridge.data.vae_splits import VaeSplits, load_vae_splits, vae_splits_fingerprint, vae_splits_recovery_fingerprint_v3
import fieldbridge.cli as fb_cli
file_inputs = {
 'split': (FROZEN_SPLIT_V3_JSON, EXPECTED_SPLIT_V3_SHA256),
 'vae_config': (FROZEN_STAGE1_VAE_CONFIG, EXPECTED_VAE_CONFIG_SHA256),
 'vae_checkpoint': (FROZEN_VAE_CHECKPOINT, EXPECTED_VAE_CHECKPOINT_SHA256),
 'gate01': (GATE01_RESULT_JSON, EXPECTED_GATE01_RESULT_SHA256),
}
for label, (path, expected) in file_inputs.items():
    path = assert_variant_a_external_path(path, repo_root=REPO_DIR)
    if not path.is_file() or sha256_file(path) != expected:
        raise RuntimeError(f'{label} missing or SHA-256 mismatch: {path}')
data_root = assert_variant_a_external_path(RETROSPECTIVE_DATA_ROOT, repo_root=REPO_DIR).resolve(strict=True)
output_root = assert_variant_a_external_path(EXTERNAL_OUTPUT_ROOT, repo_root=REPO_DIR)
if not data_root.is_dir(): raise NotADirectoryError(data_root)
output_root.mkdir(parents=True, exist_ok=True)
original_bytes = FROZEN_SPLIT_V3_JSON.read_bytes()
if hashlib.sha256(original_bytes).hexdigest() != EXPECTED_SPLIT_V3_SHA256:
    raise RuntimeError('Immutable split changed.')
original = load_vae_splits(FROZEN_SPLIT_V3_JSON)
original_membership = vae_splits_fingerprint(original)
original_recovery = vae_splits_recovery_fingerprint_v3(original)
payload = copy.deepcopy(json.loads(original_bytes.decode('utf-8')))
old_root = PureWindowsPath(REVIEWED_WINDOWS_SOURCE_ROOT)
def remap(raw):
    source = PureWindowsPath(str(raw)); lhs = tuple(x.casefold() for x in source.parts); rhs = tuple(x.casefold() for x in old_root.parts)
    if not source.is_absolute() or lhs[:len(rhs)] != rhs:
        raise ValueError(f'Path outside reviewed Windows root: {raw}')
    suffix = source.parts[len(old_root.parts):]
    if not suffix or any(x in {'', '.', '..'} for x in suffix): raise ValueError(f'Unsafe path: {raw}')
    mapped = data_root.joinpath(*suffix).resolve(strict=False)
    mapped.relative_to(data_root)
    return mapped
mapping = []
for split_name in ('train', 'validation', 'test'):
    for record in payload['splits'][split_name]:
        old = str(record['image_path']); new = remap(old); record['image_path'] = str(new)
        mapping.append({'split': split_name, 'record_identity': str(record.get('case_id', '')), 'old_path': old, 'new_path': str(new)})
mapping.sort(key=lambda x: (x['split'], x['record_identity'], x['old_path']))
mapping_sha = sha256_json(mapping)
metadata = dict(payload.get('metadata', {})); metadata['colab_path_remap'] = {'contract': 'stage2-colab-windows-root-remap-v1', 'original_split_file_sha256': EXPECTED_SPLIT_V3_SHA256, 'original_membership_fingerprint': original_membership, 'original_recovery_fingerprint_v3': original_recovery, 'reviewed_old_root': str(old_root), 'operational_new_root': str(data_root), 'remapped_record_count': len(mapping), 'mapping_identity_sha256': mapping_sha}; payload['metadata'] = metadata
def records(name): return tuple(record_from_mapping(x) for x in payload['splits'][name])
candidate = VaeSplits(train=records('train'), validation=records('validation'), test=records('test'), seed=int(payload['seed']), fractions=tuple(float(x) for x in payload['fractions']), metadata=metadata)
membership = vae_splits_fingerprint(candidate); recovery = vae_splits_recovery_fingerprint_v3(candidate)
if membership != original_membership: raise RuntimeError('Remap changed membership.')
assignments = lambda split: {name: tuple(sorted(r.case_id for r in split.records_for(name))) for name in ('train', 'validation', 'test')}
if assignments(candidate) != assignments(original): raise RuntimeError('Remap changed split assignments.')
payload['fingerprint'] = membership; payload['recovery_fingerprint_v3'] = recovery
OPERATIONAL_SPLIT = output_root / 'split_v3_colab_operational.json'
expected_bytes = (json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + '\n').encode()
if OPERATIONAL_SPLIT.exists():
    if OPERATIONAL_SPLIT.read_bytes() != expected_bytes: raise RuntimeError('Existing operational split is not exact resume.')
else: write_json_atomic(OPERATIONAL_SPLIT, payload)
splits = load_vae_splits(OPERATIONAL_SPLIT)
fit_records, fit_excluded = fb_cli._select_variant_a_retrospective_records(splits.train, split='train')
qualification_records, qualification_excluded = fb_cli._select_variant_a_retrospective_records(splits.validation, split='validation')
if set(Counter(r.domain.label for r in fit_records)) != set(all_photometry_domain_labels()) or set(Counter(r.domain.label for r in qualification_records)) != set(all_photometry_domain_labels()):
    raise RuntimeError('Complete eligible train/validation inventories must cover all 15 domains.')
inventory = []
for split_name, selected in (('train', fit_records), ('validation', qualification_records)):
    for record in selected:
        identity = fb_cli._classify_variant_a_split_record(record)
        source = Path(record.image_path).resolve(strict=True); relative = source.relative_to(data_root)
        inventory.append({'split': split_name, 'record_identity': str(record.case_id), 'record_identity_sha256': sha256_text(str(record.case_id)), 'subject_group_identity': identity.subject_group_identity, 'domain': record.domain.label, 'relative_source_path': relative.as_posix(), 'source_path_identity_sha256': sha256_text(str(record.image_path)), 'source_bytes': source.stat().st_size, 'source_file_sha256': sha256_file(source)})
inventory.sort(key=lambda x: (x['split'], x['domain'], x['record_identity']))
fit_hash = sha256_json([x for x in inventory if x['split'] == 'train']); validation_hash = sha256_json([x for x in inventory if x['split'] == 'validation'])
if fit_hash != EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256 or validation_hash != FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256:
    raise RuntimeError('Complete retrospective inventory identity mismatch.')
if FROZEN_SPLIT_V3_JSON.read_bytes() != original_bytes: raise RuntimeError('Original split was modified.')
print(json.dumps({'operational_split': str(OPERATIONAL_SPLIT), 'original_sha256': EXPECTED_SPLIT_V3_SHA256, 'operational_sha256': sha256_file(OPERATIONAL_SPLIT), 'membership': membership, 'recovery': recovery, 'mapping_sha256': mapping_sha, 'fit_count': len(fit_records), 'fit_inventory_sha256': fit_hash, 'validation_count': len(qualification_records), 'validation_inventory_sha256': validation_hash, 'domain_counts_train': dict(sorted(Counter(r.domain.label for r in fit_records).items())), 'domain_counts_validation': dict(sorted(Counter(r.domain.label for r in qualification_records).items())), 'excluded_P_train': fit_excluded, 'excluded_P_validation': qualification_excluded, 'classification_before_array_load': True, 'performance_based_selection': False}, indent=2))


In [ ]:
# Visible, durable command logging. Existing immutable outputs are validated and skipped.
def run_logged(command, log_path, operation):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        log.write(json.dumps({'operation': operation, 'commit': IMPLEMENTATION_REF, 'command': command}) + '\n')
        process = subprocess.Popen(command, cwd=REPO_DIR, env=CLI_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait(); log.write(json.dumps({'operation': operation, 'return_code': code}) + '\n'); log.flush(); os.fsync(log.fileno())
    if code: raise subprocess.CalledProcessError(code, command)
VARIANT_CONFIG = REPO_DIR / 'configs/experiment/stage2_photometry_factorization_a_v1.yaml'
PHOTOMETRY = output_root / 'stage2_photometry_factorization_v1.json'
CONTINUITY = output_root / 'stage2_photometry_continuity_reference_v2.json'
QUALIFICATION = output_root / 'stage2_photometry_variant_a_qualification_v1.json'
if not PHOTOMETRY.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'fit-stage2-photometry', '--config', str(VARIANT_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--out', str(PHOTOMETRY), '--device', 'cpu', '--log-every', '10'], output_root / 'logs/variant_a_fit.log', 'fit-stage2-photometry')
if not CONTINUITY.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'build-stage2-photometry-continuity-reference', '--gate01-result', str(GATE01_RESULT_JSON), '--evaluation-id', 'gate01-external-continuity-only-source-sha256:' + EXPECTED_GATE01_RESULT_SHA256, '--out', str(CONTINUITY)], output_root / 'logs/gate01_continuity.log', 'build-continuity-reference')
if not QUALIFICATION.exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'audit-stage2-photometry', '--config', str(VARIANT_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--artifact', str(PHOTOMETRY), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--continuity-reference', str(CONTINUITY), '--gate01-result', str(GATE01_RESULT_JSON), '--out', str(QUALIFICATION), '--device', 'cuda', '--precision', 'float32', '--log-every', '1'], output_root / 'logs/variant_a_qualification.log', 'audit-stage2-photometry')
qualification = json.loads(QUALIFICATION.read_text())
if qualification.get('canonical_latent_bank_authorized') is not True:
    raise RuntimeError({'Variant_A_qualification_failed': qualification.get('failures')})
if qualification.get('eligibility_proof', {}).get('prospective_accepted_count') != 0:
    raise RuntimeError('Variant A accepted prospective data.')
print(json.dumps({'artifact_file_sha256': sha256_file(PHOTOMETRY), 'artifact_identity': qualification['artifact_sha256'], 'qualification_file_sha256': sha256_file(QUALIFICATION), 'qualification_result_sha256': qualification['result_sha256'], 'authorized': True}, indent=2))


In [ ]:
# Streamed factored-bank filesystem/storage preflight, build, and complete source->N_d->E audit.
CANONICAL_CONFIG = REPO_DIR / 'configs/experiment/stage2_canonical_artifacts_v2.yaml'
BANK_DIR = output_root / 'photometry_factored_latent_bank_v2'
common = ['--config', str(CANONICAL_CONFIG), '--split-json', str(OPERATIONAL_SPLIT), '--photometry-artifact', str(PHOTOMETRY), '--qualification', str(QUALIFICATION), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT)]
run_logged([sys.executable, '-m', 'fieldbridge.cli', 'preflight-photometry-factored-latent-bank', *common, '--out-dir', str(BANK_DIR), '--device', 'cuda'], output_root / 'logs/factored_bank_preflight.log', 'factored-bank-preflight')
if not (BANK_DIR / 'photometry_factored_latent_bank_manifest.json').exists():
    run_logged([sys.executable, '-m', 'fieldbridge.cli', 'build-photometry-factored-latent-bank', *common, '--out-dir', str(BANK_DIR), '--device', 'cuda', '--resume', '--log-every', '1'], output_root / 'logs/factored_bank_build.log', 'factored-bank-build')
run_logged([sys.executable, '-m', 'fieldbridge.cli', 'audit-photometry-factored-latent-bank', *common, '--bank-dir', str(BANK_DIR), '--device', 'cuda', '--log-every', '1'], output_root / 'logs/factored_bank_audit.log', 'factored-bank-audit')
bank_manifest = json.loads((BANK_DIR / 'photometry_factored_latent_bank_manifest.json').read_text())
if bank_manifest['eligibility_proof']['prospective_accepted_count'] != 0 or bank_manifest['record_count'] != len(fit_records) + len(qualification_records):
    raise RuntimeError('Factored bank eligibility/inventory mismatch.')
print(json.dumps({'bank_artifact_sha256': bank_manifest['artifact_sha256'], 'record_count': bank_manifest['record_count'], 'domain_counts': bank_manifest['domain_counts'], 'canonical_persisted': bank_manifest['canonical_stream']['full_canonical_tensor_persisted'], 'support_rule': bank_manifest['operational_support_rule']['contract_version']}, indent=2))
DOMAIN_SEPARABILITY = output_root / 'stage2_factored_domain_separability_v1.json'
if not DOMAIN_SEPARABILITY.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'preflight-stage2-factored-domain-separability', '--bank-dir', str(BANK_DIR), '--out', str(DOMAIN_SEPARABILITY)], output_root / 'logs/domain_separability.log', 'factored-domain-separability')
domain_separability = json.loads(DOMAIN_SEPARABILITY.read_text()); stored = domain_separability.pop('result_sha256'); assert stored == sha256_json(domain_separability); domain_separability['result_sha256'] = stored
PAIR_FEASIBILITY = output_root / 'stage2_retrospective_pair_feasibility_v2.json'
if not PAIR_FEASIBILITY.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'audit-stage2-retrospective-pair-feasibility', '--split-json', str(OPERATIONAL_SPLIT), '--out', str(PAIR_FEASIBILITY)], output_root / 'logs/pair_feasibility.log', 'paired-feasibility')
pair_feasibility = json.loads(PAIR_FEASIBILITY.read_text()); stored = pair_feasibility.pop('result_sha256'); assert stored == sha256_json(pair_feasibility); pair_feasibility['result_sha256'] = stored
PAIRED_EVALUATION_AVAILABLE = pair_feasibility['paired_evaluation_possible'] is True
PAIRED_R_VALIDATION_MANIFEST = output_root / 'stage2_complete_R_validation_paired_manifest.json'
BASELINE_PREDICTIONS_MANIFEST = output_root / 'stage2_complete_gate01_sbv2_baselines.json'
P0006_EVALUATION_PROTOCOL = output_root / 'stage2_gate01_p0006_evaluation_only_protocol_v1.json'
LONG_RUN_EVALUATION_READINESS = output_root / 'stage2_long_run_evaluation_readiness_v2.json'
if PAIRED_EVALUATION_AVAILABLE:
    if not R_PAIRED_EVALUATION_ARCHIVE_RAW: raise RuntimeError('Genuine R/validation pairs exist. Supply the reviewed R-paired producer-export archive; the repository importer constructs every evaluator manifest.')
    R_PAIRED_EVALUATION_ARCHIVE = assert_variant_a_external_path(Path(R_PAIRED_EVALUATION_ARCHIVE_RAW).expanduser(), repo_root=REPO_DIR)
    if not LONG_RUN_EVALUATION_READINESS.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'import-stage2-retrospective-paired-evaluation', '--feasibility', str(PAIR_FEASIBILITY), '--archive-root', str(R_PAIRED_EVALUATION_ARCHIVE), '--photometry-artifact', str(PHOTOMETRY), '--authorization-reference', 'complete-R-validation-feasibility:' + pair_feasibility['result_sha256'], '--out-dir', str(output_root)], output_root / 'logs/R_paired_evaluation_import.log', 'import-R-paired-evaluation')
else:
    print(json.dumps({'paired_R_validation_evaluation': 'not_possible', 'reason': pair_feasibility['failure_instruction'], 'fabricated_pairs': False, 'fallback': 'sealed_P0006_evaluation_only_after_frozen_validation_plan'}, indent=2))
print(json.dumps({'domain_separability_sha256': domain_separability['result_sha256'], 'validation_accuracy': domain_separability['validation_accuracy'], 'per_domain': domain_separability['per_domain'], 'paired_evaluation_possible': PAIRED_EVALUATION_AVAILABLE, 'paired_feasibility_sha256': pair_feasibility['result_sha256'], 'StarGAN_control_claim': False}, indent=2))


In [ ]:
# Mandatory 200-step full-objective pilot with gradient, GAN, runtime and validation gates.
UNIFIED_CONFIG = REPO_DIR / 'configs/experiment/stage2_unified_full_retrospective_v4.yaml'
from fieldbridge.training.stage2_unified import find_latest_stage2_selection_receipt
import yaml
PILOT_DIR = output_root / 'unified_full_objective_pilot_200'
pilot_config = yaml.safe_load(UNIFIED_CONFIG.read_text()); pilot_config['training']['pilot']['gpu_hourly_cost_usd'] = GPU_HOURLY_COST_USD
PILOT_CONFIG = PILOT_DIR / 'resolved_config.json'; PILOT_DIR.mkdir(parents=True, exist_ok=True)
if PILOT_CONFIG.exists() and json.loads(PILOT_CONFIG.read_text()) != pilot_config: raise RuntimeError('Pilot config/cost changed for exact resume.')
if not PILOT_CONFIG.exists(): write_json_atomic(PILOT_CONFIG, pilot_config)
PILOT_CHECKPOINTS = PILOT_DIR / 'checkpoints'; PILOT_HISTORY = PILOT_DIR / 'history.jsonl'
pilot_checkpoint = PILOT_CHECKPOINTS / 'stage2_unified_full_step000000200.pt'
if not pilot_checkpoint.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'train-stage2-unified', '--config', str(PILOT_CONFIG), '--bank-dir', str(BANK_DIR), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--checkpoint-dir', str(PILOT_CHECKPOINTS), '--history-jsonl', str(PILOT_HISTORY), '--steps', '200', '--pilot-steps', '200', '--device', 'cuda'], PILOT_DIR / 'diagnostic.log', 'full-objective-pilot-200')
pilot_events = [json.loads(line) for line in PILOT_HISTORY.read_text().splitlines() if line.strip()]
pilot = next((item['pilot'] for item in reversed(pilot_events) if item.get('event') == 'full_objective_pilot'), None)
validation = next((item['validation'] for item in reversed(pilot_events) if item.get('event') == 'unpaired_validation'), None)
if pilot is None or pilot.get('status') != 'pass' or pilot.get('steps') != 200 or validation is None or validation.get('complete_inventory_used') is not True: raise RuntimeError({'pilot': pilot, 'validation': validation})
PILOT_SELECTION_RECEIPT, pilot_selection = find_latest_stage2_selection_receipt(PILOT_CHECKPOINTS, variant='full', require_complete=True)
if pilot_selection['validation_plan_sha256'] != validation['validation_plan_sha256']: raise RuntimeError('Pilot checkpoints and validation event do not share the frozen validation plan.')
FROZEN_VALIDATION_PLAN_SHA256 = pilot_selection['validation_plan_sha256']
FROZEN_VALIDATION_PLAN = PILOT_CHECKPOINTS / 'stage2_unified_validation_plan_v2.json'
validation_plan_payload = json.loads(FROZEN_VALIDATION_PLAN.read_text()); plan_hash = validation_plan_payload.pop('validation_plan_sha256'); assert plan_hash == sha256_json(validation_plan_payload) == FROZEN_VALIDATION_PLAN_SHA256; validation_plan_payload['validation_plan_sha256'] = plan_hash
if validation_plan_payload.get('required_directed_domain_cell_count') != 60 or set(validation_plan_payload.get('directed_domain_cell_counts', {}).values()) == {0}: raise RuntimeError('Frozen validation plan lacks complete 60-cell coverage.')
if not PAIRED_EVALUATION_AVAILABLE:
    if not P0006_EVALUATION_PROTOCOL.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'import-stage2-gate01-p0006-evaluation', '--archive-root', str(GATE01_PRIVATE_ARCHIVE_ROOT), '--expected-gate01-result-sha256', EXPECTED_GATE01_RESULT_SHA256, '--bank-dir', str(BANK_DIR), '--validation-plan', str(FROZEN_VALIDATION_PLAN), '--out', str(P0006_EVALUATION_PROTOCOL)], output_root / 'logs/P0006_evaluation_import.log', 'import-P0006-evaluation-only')
    if not LONG_RUN_EVALUATION_READINESS.exists(): run_logged([sys.executable, '-m', 'fieldbridge.cli', 'seal-stage2-long-run-evaluation-readiness', '--feasibility', str(PAIR_FEASIBILITY), '--p0006-evaluation-protocol', str(P0006_EVALUATION_PROTOCOL), '--out', str(LONG_RUN_EVALUATION_READINESS)], output_root / 'logs/long_run_evaluation_readiness.log', 'seal-long-run-evaluation-readiness-P0006')
EVALUATION_ARGS = (['--paired-manifest', str(PAIRED_R_VALIDATION_MANIFEST), '--baseline-predictions', str(BASELINE_PREDICTIONS_MANIFEST)] if PAIRED_EVALUATION_AVAILABLE else ['--p0006-evaluation-protocol', str(P0006_EVALUATION_PROTOCOL)])
print(json.dumps({'pilot_checkpoint': str(pilot_checkpoint), 'pilot_selection_receipt': str(PILOT_SELECTION_RECEIPT), 'validation_plan_sha256': FROZEN_VALIDATION_PLAN_SHA256, 'selection_rule_sha256': pilot_selection['selection_rule_sha256'], 'loss_behavior': pilot['loss_behavior'], 'term_gradients': pilot['term_gradient_norms'], 'critic': pilot['critic'], 'runtime_projection': pilot['runtime'], 'validation_selection': validation}, indent=2))


In [ ]:
# Evaluation readiness is verified before the first long-run authorization is consulted.
if not LONG_RUN_EVALUATION_READINESS.is_file():
    raise RuntimeError('100k training is blocked: neither a complete genuine R/validation path nor the sealed P:0006 evaluation-only protocol is ready.')
evaluation_readiness = json.loads(LONG_RUN_EVALUATION_READINESS.read_text()); readiness_hash = evaluation_readiness.pop('readiness_sha256', None)
if readiness_hash != sha256_json(evaluation_readiness) or evaluation_readiness.get('long_run_authorized_by_evaluation_path') is not True: raise RuntimeError('Long-run evaluation-readiness receipt failed closed.')
expected_role = 'complete_genuine_paired_R_validation' if PAIRED_EVALUATION_AVAILABLE else 'sealed_held_out_P0006_final_evaluation_only'
if evaluation_readiness.get('evaluation_role') != expected_role or evaluation_readiness.get('prospective_training_or_model_selection_use', False) is not False: raise RuntimeError('Evaluation readiness chose the wrong scientific role.')
evaluation_readiness['readiness_sha256'] = readiness_hash
# First long-run authorization: launch/resume the strongest full model only.
AUTHORIZE_LONG_FULL_MODEL = False
if AUTHORIZE_LONG_FULL_MODEL is not True:
    raise PermissionError('Review the 200-step pilot, then set AUTHORIZE_LONG_FULL_MODEL=True. No ablation is launched by this flag.')
import yaml
base_config = yaml.safe_load(UNIFIED_CONFIG.read_text())
FULL_RUN_DIR = output_root / 'unified_training' / 'full'; FULL_CHECKPOINT_DIR = FULL_RUN_DIR / 'checkpoints'; FULL_HISTORY = FULL_RUN_DIR / 'history.jsonl'; FULL_CONFIG = FULL_RUN_DIR / 'resolved_config.json'; FULL_RUN_DIR.mkdir(parents=True, exist_ok=True)
if FULL_CONFIG.exists() and json.loads(FULL_CONFIG.read_text()) != base_config: raise RuntimeError('Full-model config changed for exact resume.')
if not FULL_CONFIG.exists(): write_json_atomic(FULL_CONFIG, base_config)
full_candidates = sorted(FULL_CHECKPOINT_DIR.glob('stage2_unified_full_step*.pt')) if FULL_CHECKPOINT_DIR.exists() else []
full_command = [sys.executable, '-m', 'fieldbridge.cli', 'train-stage2-unified', '--config', str(FULL_CONFIG), '--bank-dir', str(BANK_DIR), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--checkpoint-dir', str(FULL_CHECKPOINT_DIR), '--history-jsonl', str(FULL_HISTORY), '--device', 'cuda']
if full_candidates: full_command += ['--resume-from', str(full_candidates[-1])]
run_logged(full_command, FULL_RUN_DIR / 'diagnostic.log', 'train-full-only')
FULL_SELECTION_RECEIPT, full_selection = find_latest_stage2_selection_receipt(FULL_CHECKPOINT_DIR, variant='full', require_complete=True)
if full_selection['validation_plan_sha256'] != FROZEN_VALIDATION_PLAN_SHA256: raise RuntimeError('Full model did not use the frozen pilot validation plan.')
FULL_CHECKPOINT = Path(full_selection['best_checkpoint'])
FULL_FINAL_CHECKPOINT = Path(full_selection['checkpoint_hashes']['final']['path'])
FULL_EVALUATION_DIR = output_root / 'unified_retrospective_evaluation_full_only'
full_eval = [sys.executable, '-m', 'fieldbridge.cli', 'eval-stage2-unified', '--config', str(UNIFIED_CONFIG), '--bank-dir', str(BANK_DIR), '--selection-receipt', str(FULL_SELECTION_RECEIPT), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--photometry-artifact', str(PHOTOMETRY), *EVALUATION_ARGS, '--out', str(FULL_EVALUATION_DIR), '--device', 'cuda', '--integration-steps', '20', '--solver', 'heun', '--resume']
run_logged(full_eval, output_root / 'logs/unified_full_evaluation.log', 'unified-full-selected-best-evaluation')
full_result = json.loads((FULL_EVALUATION_DIR / 'result.json').read_text())
print(json.dumps({'selected_best_checkpoint': str(FULL_CHECKPOINT), 'selected_best_sha256': full_selection['checkpoint_hashes']['best']['file_sha256'], 'final_checkpoint_diagnostic_only': str(FULL_FINAL_CHECKPOINT), 'final_checkpoint_sha256': full_selection['checkpoint_hashes']['final']['file_sha256'], 'selection_receipt': str(FULL_SELECTION_RECEIPT), 'full_evaluation': full_result, 'backward_ablations_launched': False}, indent=2))


In [ ]:
# Separate post-review authorization for backward ablations; never launched by the full-model flag.
AUTHORIZE_BACKWARD_ABLATIONS_AFTER_FULL_REVIEW = False
ABLATION_CHECKPOINTS = {}
ABLATION_FINAL_CHECKPOINTS = {}
ABLATION_SELECTION_RECEIPTS = {}
ABLATION_EVALUATION_DIR = None
if AUTHORIZE_BACKWARD_ABLATIONS_AFTER_FULL_REVIEW is not True:
    print('Intentional stop: review the strongest full-model pilot/evaluation before authorizing backward ablations.')
else:
    if not FULL_FINAL_CHECKPOINT.name.endswith('step000100000.pt'): raise RuntimeError('Full 100k-step model must complete before any backward ablation.')
    ablations = {'no_graph': {'graph': 0.0}, 'no_anatomy_graph': {'anatomy': 0.0, 'graph': 0.0}, 'no_adversarial_domain': {'adversarial': 0.0, 'domain': 0.0}, 'sb_identity_only': {'anatomy': 0.0, 'graph': 0.0, 'adversarial': 0.0, 'domain': 0.0}, 'sb_only': {'identity': 0.0, 'anatomy': 0.0, 'graph': 0.0, 'adversarial': 0.0, 'domain': 0.0}}
    for variant, overrides in ablations.items():
        cfg = copy.deepcopy(base_config); cfg['training']['variant'] = variant; cfg['training']['loss_weights'].update(overrides)
        run_dir = output_root / 'unified_training' / variant; checkpoint_dir = run_dir / 'checkpoints'; history = run_dir / 'history.jsonl'; config_path = run_dir / 'resolved_config.json'; run_dir.mkdir(parents=True, exist_ok=True)
        if config_path.exists() and json.loads(config_path.read_text()) != cfg: raise RuntimeError(f'Ablation config changed for exact resume: {variant}')
        if not config_path.exists(): write_json_atomic(config_path, cfg)
        candidates = sorted(checkpoint_dir.glob(f'stage2_unified_{variant}_step*.pt')) if checkpoint_dir.exists() else []
        command = [sys.executable, '-m', 'fieldbridge.cli', 'train-stage2-unified', '--config', str(config_path), '--bank-dir', str(BANK_DIR), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--checkpoint-dir', str(checkpoint_dir), '--history-jsonl', str(history), '--device', 'cuda']
        if candidates: command += ['--resume-from', str(candidates[-1])]
        run_logged(command, run_dir / 'diagnostic.log', f'train-{variant}')
        receipt_path, receipt = find_latest_stage2_selection_receipt(checkpoint_dir, variant=variant, require_complete=True)
        if receipt['validation_plan_sha256'] != FROZEN_VALIDATION_PLAN_SHA256: raise RuntimeError(f'Ablation {variant} used a different validation plan.')
        ABLATION_SELECTION_RECEIPTS[variant] = receipt_path
        ABLATION_CHECKPOINTS[variant] = Path(receipt['best_checkpoint'])
        ABLATION_FINAL_CHECKPOINTS[variant] = Path(receipt['checkpoint_hashes']['final']['path'])
    ABLATION_EVALUATION_DIR = output_root / 'unified_retrospective_evaluation_all_trained_ablations'
    command = [sys.executable, '-m', 'fieldbridge.cli', 'eval-stage2-unified', '--config', str(UNIFIED_CONFIG), '--bank-dir', str(BANK_DIR), '--selection-receipt', str(FULL_SELECTION_RECEIPT), '--sb-only-checkpoint', str(ABLATION_CHECKPOINTS['sb_only']), '--vae-config', str(FROZEN_STAGE1_VAE_CONFIG), '--vae-checkpoint', str(FROZEN_VAE_CHECKPOINT), '--photometry-artifact', str(PHOTOMETRY), *EVALUATION_ARGS, '--out', str(ABLATION_EVALUATION_DIR), '--device', 'cuda', '--integration-steps', '20', '--solver', 'heun', '--resume']
    for variant, checkpoint in sorted(ABLATION_CHECKPOINTS.items()):
        if variant != 'sb_only': command += ['--ablation-checkpoint', f'{variant}={checkpoint}']
    run_logged(command, output_root / 'logs/unified_all_ablation_evaluation.log', 'evaluate-every-trained-ablation-selected-best')
    print(json.dumps({'ablation_selected_best_checkpoints': {name: str(path) for name, path in ABLATION_CHECKPOINTS.items()}, 'ablation_final_checkpoints_diagnostic_only': {name: str(path) for name, path in ABLATION_FINAL_CHECKPOINTS.items()}, 'selection_receipts': {name: str(path) for name, path in ABLATION_SELECTION_RECEIPTS.items()}, 'paired_evaluation_dir': str(ABLATION_EVALUATION_DIR)}, indent=2))


In [ ]:
# Seal a compact paper-review index without touching repository files.
evidence = {'contract': 'stage2-unified-colab-evidence-index-v4', 'implementation_commit': IMPLEMENTATION_REF, 'sealed_inputs': {label: {'path': str(path), 'file_sha256': sha256_file(path)} for label, (path, _) in file_inputs.items()}, 'operational_split_sha256': sha256_file(OPERATIONAL_SPLIT), 'photometry_artifact_sha256': sha256_file(PHOTOMETRY), 'qualification_sha256': sha256_file(QUALIFICATION), 'bank_manifest_sha256': sha256_file(BANK_DIR / 'photometry_factored_latent_bank_manifest.json'), 'domain_separability_sha256': sha256_file(DOMAIN_SEPARABILITY), 'paired_feasibility_sha256': sha256_file(PAIR_FEASIBILITY), 'evaluation_readiness_sha256': readiness_hash, 'evaluation_role': evaluation_readiness['evaluation_role'], 'paired_R_validation_evaluation_possible': PAIRED_EVALUATION_AVAILABLE, 'validation_plan_sha256': FROZEN_VALIDATION_PLAN_SHA256, 'validation_directed_domain_cell_count': 60, 'pilot_checkpoint_sha256': sha256_file(pilot_checkpoint), 'pilot_selection_receipt_sha256': sha256_file(PILOT_SELECTION_RECEIPT), 'full_selected_best_checkpoint_sha256': sha256_file(FULL_CHECKPOINT), 'full_final_checkpoint_sha256': sha256_file(FULL_FINAL_CHECKPOINT), 'full_selection_receipt_sha256': sha256_file(FULL_SELECTION_RECEIPT), 'full_evaluation_result_sha256': sha256_file(FULL_EVALUATION_DIR / 'result.json') if (FULL_EVALUATION_DIR / 'result.json').exists() else None, 'ablation_selected_best_checkpoint_sha256': {name: sha256_file(path) for name, path in sorted(ABLATION_CHECKPOINTS.items())}, 'ablation_final_checkpoint_sha256': {name: sha256_file(path) for name, path in sorted(ABLATION_FINAL_CHECKPOINTS.items())}, 'ablation_selection_receipt_sha256': {name: sha256_file(path) for name, path in sorted(ABLATION_SELECTION_RECEIPTS.items())}, 'ablation_evaluation_result_sha256': sha256_file(ABLATION_EVALUATION_DIR / 'result.json') if ABLATION_EVALUATION_DIR and (ABLATION_EVALUATION_DIR / 'result.json').exists() else None, 'prospective_training_or_model_selection': False, 'held_out_P0006_final_evaluation': not PAIRED_EVALUATION_AVAILABLE, 'descriptor_coupling': False, 'learned_disentanglement_claim': False}
if PAIRED_EVALUATION_AVAILABLE: evidence['sealed_inputs']['R_paired_evaluation_archive_identity_sha256'] = sha256_text(str(R_PAIRED_EVALUATION_ARCHIVE.resolve()))
else: evidence['sealed_inputs']['P0006_evaluation_protocol'] = {'path': str(P0006_EVALUATION_PROTOCOL), 'file_sha256': sha256_file(P0006_EVALUATION_PROTOCOL)}
evidence['evidence_sha256'] = sha256_json(evidence)
evidence_path = output_root / 'stage2_unified_evidence_index.json'
if evidence_path.exists():
    if json.loads(evidence_path.read_text()) != evidence: raise RuntimeError('Existing evidence index mismatch.')
else: write_json_atomic(evidence_path, evidence)
if FROZEN_SPLIT_V3_JSON.read_bytes() != original_bytes or git_text('status', '--porcelain'):
    raise RuntimeError('Immutable split or detached checkout changed.')
print(json.dumps(evidence, indent=2))
